# **Feature Extraction** for NeoScreen
## Extracting audio features from all three datasets: **ICBHI 2017**, **HLS-CMDS**, **SPRSound**

## 1. Imports

In [6]:
import os
import sys
import pandas as pd
import numpy as np
from pathlib import Path
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import GroupShuffleSplit

# Add project root to path
sys.path.append('..')

from EDA.audio_features import AudioFeatureExtractor

print("Imports successful")

Imports successful


## 2. Define path

In [7]:
sprsound_path = Path("../../sound_data/sprsound")
output_dir = Path("../sound_data/features")
output_dir.mkdir(parents=True, exist_ok=True)

def find_sprsound_wavs(base_path):
    wav_files = []
    # os.walk handles nested year folders (2022, 2023, etc.)
    for root, _, files in os.walk(base_path):
        for file in files:
            if file.lower().endswith('.wav'):
                wav_files.append(Path(root) / file)
    return wav_files

## 3. Initialize Feature Extractor

In [8]:
extractor = AudioFeatureExtractor()

if sprsound_path.exists():
    sprsound_files = find_sprsound_wavs(sprsound_path)
    print(f"Found {len(sprsound_files)} SPRSound files.")
    
    if len(sprsound_files) > 0:
        print("Starting batch extraction (this may take a few minutes)...")
        # Extract features
        df_sprsound = extractor.extract_batch(
            sprsound_files, 
            output_csv=output_dir / "sprsound_features.csv"
        )
        print(f"Extraction complete! Shape: {df_sprsound.shape}")
    else:
        print("No .wav files found in the folder.")
else:
    print(f"SPRSound folder not found at: {sprsound_path.absolute()}")

AudioFeatureExtractor initialized with sr=4000, n_mfcc=13
Found 6567 SPRSound files.
Starting batch extraction (this may take a few minutes)...

Processing 6567 files...
   Progress: 0/6567 (Success: 0, Failed: 0)
   Progress: 10/6567 (Success: 10, Failed: 0)
   Progress: 20/6567 (Success: 20, Failed: 0)
   Progress: 30/6567 (Success: 30, Failed: 0)
   Progress: 40/6567 (Success: 40, Failed: 0)
   Progress: 50/6567 (Success: 50, Failed: 0)
   Progress: 60/6567 (Success: 60, Failed: 0)
   Progress: 70/6567 (Success: 70, Failed: 0)
   Progress: 80/6567 (Success: 80, Failed: 0)
   Progress: 90/6567 (Success: 90, Failed: 0)
   Progress: 100/6567 (Success: 100, Failed: 0)
   Progress: 110/6567 (Success: 110, Failed: 0)
   Progress: 120/6567 (Success: 120, Failed: 0)
   Progress: 130/6567 (Success: 130, Failed: 0)
   Progress: 140/6567 (Success: 140, Failed: 0)
   Progress: 150/6567 (Success: 150, Failed: 0)
   Progress: 160/6567 (Success: 160, Failed: 0)
   Progress: 170/6567 (Success: 170,

## 4. Extract all patient IDs

In [9]:
# Create df_all from our newly extracted features
df_all = df_sprsound.copy()

# In SPRSound, filenames are typically structured as 'PatientID_Year_Location.wav'
# We extract the first segment as the unique Patient ID
df_all['patient_id'] = df_all['filename'].apply(lambda x: str(x).split('_')[0])

# Add a marker for the dataset
df_all['dataset'] = 'SPRSound'

print(f"Unique Neonatal Patients: {df_all['patient_id'].nunique()}")
print(f"Sample of extracted IDs: {df_all['patient_id'].unique()[:5]}")

Unique Neonatal Patients: 869
Sample of extracted IDs: ['40512331' '40638274' '40672181' '40728258' '40794666']


## 5. Train/Test Split

In [10]:
# 80% of PATIENTS for training, 20% for testing
gss = GroupShuffleSplit(n_splits=1, train_size=0.8, random_state=42)

# Use 'patient_id' as the group
train_idx, test_idx = next(gss.split(df_all, groups=df_all['patient_id']))

df_train = df_all.iloc[train_idx]
df_test = df_all.iloc[test_idx]

print(f"Splitting complete:")
print(f"- Training: {len(df_train)} recordings from {df_train['patient_id'].nunique()} patients")
print(f"- Testing: {len(df_test)} recordings from {df_test['patient_id'].nunique()} patients")

# Verify no overlap
overlap = set(df_train['patient_id']).intersection(set(df_test['patient_id']))
assert len(overlap) == 0, "Data Leakage detected! Patients exist in both sets."

Splitting complete:
- Training: 5225 recordings from 695 patients
- Testing: 1342 recordings from 174 patients
